# Expression Quality Control (Part 2)

This is a template notebook for performing the final quality control on your organism's expression data. This requires a curated metadata sheet.

## Setup 

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from os import path
from scipy import stats
from tqdm.notebook import tqdm

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
sns.set_style('ticks')

### Inputs

In [ ]:
#logTPM_file = path.join('..','data','raw_data','log_tpm.csv') # Enter log-TPM filename here
#all_metadata_file = path.join('..','data','interim','metadata_qc_part1_all.tsv') # Enter full metadata filename here
#metadata_file = path.join('..','data','interim','metadata_qc_part1.tsv') # Enter curated metadata filename here


logTPM_file = path.join('../data/interim', "log_tpm.tsv") # Enter log-TPM filename here
all_metadata_file = path.join('../data/','interim/','metadata_qc_part1_all.tsv') # Enter full metadata filename here
metadata_file = path.join("../data/interim/metadata_qc_part1_curated.tsv") # Enter curated metadata filename here

### Load expression data

In [ ]:
DF_log_tpm = pd.read_csv(logTPM_file,index_col=[0], sep='\t').fillna(0)
print('Number of genes:',DF_log_tpm.shape[0])
print('Number of samples:',DF_log_tpm.shape[1])
DF_log_tpm.head()

### Load metadata

In [ ]:
DF_metadata = pd.read_csv(metadata_file,index_col=0, sep='\t')
print('Number of samples with curated metadata:',DF_metadata.shape[0])
DF_metadata.head()

In [ ]:
DF_metadata["skip"].value_counts()

In [ ]:
DF_metadata_all = pd.read_csv(all_metadata_file,index_col=0,sep='\t')

## Remove samples due to poor metadata

After curation, some samples either did not have enough replicates or metadata to warrant inclusion in this database.

In [ ]:
DF_metadata_passed_step4 = DF_metadata[~DF_metadata.skip.fillna(False)].copy()
print('New number of samples with curated metadata:',DF_metadata_passed_step4.shape[0])
DF_metadata_passed_step4

### Check curation
Since manual curation is error-prone, we want to make sure that all samples have labels for their project and condition. In addition, there should only be one reference condition in each project, and it should be in the project itself.

Any samples that fail these checks will be printed below.

In [ ]:
assert(DF_metadata_passed_step4.project.notnull().all())
assert(DF_metadata_passed_step4.condition.notnull().all())

for name,group in DF_metadata_passed_step4.groupby('project'):
    ref_cond = group.reference_condition.unique()
    
    # Ensure that there is only one reference condition per project
    if not len(ref_cond) == 1:
        print('Multiple reference conditions for:, name')
    
    # Ensure the reference condition is in fact in the project
    ref_cond = ref_cond[0]
    if not ref_cond in group.condition.tolist():
        print('Reference condition not in project:', name)

Next, make a new column called ``full_name`` that gives every experimental condition a unique, human-readable identifier.

In [ ]:
DF_metadata_passed_step4['full_name'] = DF_metadata_passed_step4['project'].str.cat(DF_metadata_passed_step4['condition'],sep=':')

In [ ]:
DF_metadata_passed_step4['full_name'].value_counts().sum()

### Remove samples with only one replicate

First, find sample names that have at least two replicates.

In [ ]:
counts = DF_metadata_passed_step4.full_name.value_counts()
keep_samples = counts[counts >= 2].index
print(len(keep_samples))

Only keep these samples

In [ ]:
DF_metadata_passed_step4 = DF_metadata_passed_step4[DF_metadata_passed_step4.full_name.isin(keep_samples)]
print('New number of samples with curated metadata:',DF_metadata_passed_step4.shape[0])
DF_metadata_passed_step4

### Save this information to the full metadata dataframe

In [ ]:
DF_metadata_all['passed_curation'] = DF_metadata_all.index.isin(DF_metadata_passed_step4.index)

## Check correlations between replicates

### Remove failed data from log_tpm files

In [ ]:
DF_log_tpm = DF_log_tpm[DF_metadata_passed_step4.index]

### Compute Pearson R Score

Biological replicates should have a Pearson R correlation above 0.95. For samples with more than 2 replicates, the replicates must have R >= 0.95 with at least one other replicate or it will be dropped. The correlation threshold can be changed below:

In [ ]:
rcutoff = 0.95

The following code computes correlations between all samples and collects correlations between replicates and non-replicates.

In [ ]:
rep_corrs = {}
rand_corrs = {}

num_comparisons = len(DF_metadata_passed_step4)*(len(DF_metadata_passed_step4)-1)/2

for exp1,exp2 in tqdm(itertools.combinations(DF_metadata_passed_step4.index,2),total=num_comparisons):
    if DF_metadata_passed_step4.loc[exp1,'full_name'] == DF_metadata_passed_step4.loc[exp2,'full_name']:
        rep_corrs[(exp1,exp2)] = stats.pearsonr(DF_log_tpm[exp1],DF_log_tpm[exp2])[0]
    else:
        rand_corrs[(exp1,exp2)] = stats.pearsonr(DF_log_tpm[exp1],DF_log_tpm[exp2])[0]

Correlations can be plotted on a histogram

In [ ]:
fig,ax = plt.subplots(figsize=(5,3))
ax2 = ax.twinx()
ax2.hist(rep_corrs.values(),bins=50,range=(0.2,1),alpha=0.8,color='green',linewidth=0)
ax.hist(rand_corrs.values(),bins=50,range=(0.2,1),alpha=0.8,color='blue',linewidth=0)
ax.set_title('Pearson R correlation between experiments',fontsize=14)
ax.set_xlabel('Pearson R correlation',fontsize=14)
ax.set_ylabel('Different Conditions',fontsize=14)
ax2.set_ylabel('Known Replicates',fontsize=14)

med_corr = np.median([v for k,v in rep_corrs.items()])
med_corr_rand = np.median([v for k,v in rand_corrs.items()])
print('Median Pearson R between replicates: {:.2f}'.format(med_corr))
plt.text(.05,360,f'Replicate Median: {med_corr:.2f}\nNon-rep. Median: {med_corr_rand:.2f}')
plt.xlim([0,1])

# Vertical reference line
ax.axvline(
    x=0.85,
    color='black',
    linestyle='--',
    linewidth=2,
    alpha=0.8
)

# Label
ax.text(
    0.852,                     # slightly to the right of the line
    ax.get_ylim()[1] * 0.95,   # near the top of the axis
    'E. coli MG1655\nNon-Rep. Corr.',
    rotation=90,
    va='top',
    ha='left',
    fontsize=11,
    color='black'
)

plt.savefig('figures/images/fig1_replicate_corr_hist.svg', format = 'svg')

Remove samples without any high-correlation replicates

In [ ]:
dissimilar = []
for idx, grp in DF_metadata_passed_step4.groupby('full_name'):
    ident = np.identity(len(grp))
    corrs = (DF_log_tpm[grp.index].corr() - ident).max()
    dissimilar.extend(corrs[corrs<rcutoff].index)

# Save this information in both the original metadata dataframe and the new metadata dataframe
DF_metadata_all['passed_replicate_correlations'] = ~DF_metadata_all.index.isin(dissimilar)
DF_metadata_passed_step4['passed_replicate_correlations'] = ~DF_metadata_passed_step4.index.isin(dissimilar)

In [ ]:
DF_metadata_passed_step4[DF_metadata_passed_step4.project == 'ou_aggregate_toxicity']

In [ ]:
DF_metadata_final = DF_metadata_passed_step4[DF_metadata_passed_step4['passed_replicate_correlations']]
print('# Samples that passed replicate correlations:',len(DF_metadata_final))

In [ ]:
DF_metadata_final["project"].value_counts()

## Check that reference conditions still exist
If a reference condition was removed due to poor replicate correlations, a new reference condition needs to be defined.

Again, any samples that fail these checks will be printed below.

In [ ]:
project_exprs = []
for name,group in DF_metadata_final.groupby('project'):
    
    # Get reference condition
    ref_cond = group.reference_condition.iloc[0]
    
    # Ensure the reference condition is still in the project
    if ref_cond not in group.condition.tolist():
        print('Reference condition missing from:', name)
    
    # Check that each project has at least two conditions (a reference and at least one test condition)
    if len(group.condition.unique()) <= 1:
        print('Only one condition in:', name)

In [ ]:
# drop bad projects
bad_projects = ['MIT_splicing', 'ou_aggregate_toxicity']
DF_metadata_final = DF_metadata_final[~DF_metadata_final.project.isin(bad_projects)]

If necessary, choose a new condition for failed projects and re-run notebook.

## Deseq2 Normalization of Counts for Downstream Use

In [ ]:
counts_path = path.join('..', 'data', 'raw_data', 'salmon.merged.gene_counts_length_scaled.tsv')

DF_counts = pd.read_csv(counts_path, index_col=0, sep='\t').drop('gene_name', axis=1)
DF_counts_full = DF_counts[[x for x in DF_metadata.index if type(x) == type("")]]
DF_counts = DF_counts[DF_metadata_final.index]
DF_counts.to_csv(path.join('..', 'data', 'interim', 'counts_data.csv'))
for index in DF_counts.index: # remove 'gene-' from each gene 
    DF_counts.rename(index={index:index.strip('gene-')},inplace=True)
    DF_counts_full.rename(index={index:index.strip('gene-')},inplace=True)

In [ ]:
# use this cell if you want to decompose on protein-coding genes only (see notebook 0)
genes = []
with open('../data/sequence_files/protein_coding_genes.txt', 'r') as genes_file:
    for line in genes_file:
        genes.append(line.strip())

DF_counts = DF_counts.loc[genes]
DF_counts_full = DF_counts_full.loc[genes]

In [ ]:
# normalize the data using pydeseq2's median of ratios method
from pydeseq2.preprocessing import deseq2_norm

DF_normalizedCounts, size_factors = deseq2_norm(DF_counts.T)
DF_normalizedCounts = DF_normalizedCounts.T

DF_log_normalizedCounts = DF_normalizedCounts + 1
DF_log_normalizedCounts = np.log2(DF_log_normalizedCounts)
DF_log_normalizedCounts

In [ ]:
DF_log_normalizedCounts.iloc[:,0].sort_values()

In [ ]:
size_factors

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.patches as patches

def global_clustering(data, threshold=0.3, xticklabels=False, yticklabels=False, figsize=(9,9)):
    
    # Retrieve clusters using fcluster 
    corr = data.corr()
    corr.fillna(0,inplace=True)
    dist = sch.distance.pdist(corr)
    link = sch.linkage(dist, method='complete')
    clst = pd.DataFrame(index=data.columns)
    clst['cluster'] = sch.fcluster(link, threshold * dist.max(), 'distance')

    # Get colors for each cluster
    cm = plt.cm.get_cmap('tab20')
    cluster_colors = dict(zip(clst.cluster.unique(), cm.colors + cm.colors))
    clst['color'] = clst.cluster.map(cluster_colors)

    print('Number of cluster: ', len(cluster_colors))
    
    legend_items = [patches.Patch(color=c, label=l) for l,c in cluster_colors.items()]
    
    sns.set(rc={'figure.facecolor':'white'})
    
    clst_map = sns.clustermap(data.corr(), 
                              figsize=figsize, 
                              row_linkage=link, 
                              col_linkage=link, 
                              col_colors=clst.color,
                              yticklabels=yticklabels, 
                              xticklabels=xticklabels,
                              vmin=0, 
                              vmax=1)
    
    legend = clst_map.ax_heatmap.legend(loc='upper left', 
                                        bbox_to_anchor=(1.01,0.85), 
                                        handles=legend_items,
                                        frameon=True)
    
    legend.set_title(title='Clusters',prop={'size':10})
    
    return clst['cluster']

In [ ]:
clusters = global_clustering(DF_log_normalizedCounts)

## Normalize dataset to reference conditions

In [ ]:
DF_log_tpm_final = DF_log_tpm[DF_metadata_final.index]

In [ ]:
# normalize log tpm data
project_exprs = []
for name,group in DF_metadata_final.groupby('project'):
    
    # Get reference condition
    ref_cond = group.reference_condition.iloc[0]
    
    # Get reference condition sample ids
    ref_samples = group[group.condition == ref_cond].index
    
    # Get reference condition expression
    ref_expr = DF_log_tpm_final[ref_samples].mean(axis=1)
    
    # Subtract reference expression from project
    project_exprs.append(DF_log_tpm_final[group.index].sub(ref_expr,axis=0))

DF_log_tpm_norm = pd.concat(project_exprs,axis=1)

In [ ]:
# normalize log normalized counts data to references
project_exprs = []
for name,group in DF_metadata_final.groupby('project'):
    
    # Get reference condition
    ref_cond = group.reference_condition.iloc[0]
    
    # Get reference condition sample ids
    ref_samples = group[group.condition == ref_cond].index
    
    # Get reference condition expression
    ref_expr = DF_log_normalizedCounts[ref_samples].mean(axis=1)
    
    # Subtract reference expression from project
    project_exprs.append(DF_log_normalizedCounts[group.index].sub(ref_expr,axis=0))

DF_log_normalizedCounts_norm = pd.concat(project_exprs,axis=1)

In [ ]:
DF_log_normalizedCounts_norm.isna().any().value_counts()

In [ ]:
DF_log_tpm_norm.isna().any().value_counts()

## Save final datasets

In [ ]:
logTPM_qc_file = path.join('../data','processed_data','log_tpm.csv')
logTPM_norm_file = path.join('../data','processed_data','log_tpm_norm.csv')
logNormalizedCounts_qc_file = path.join('../data','processed_data','log_normalizedCounts.csv')
logNormalizedCounts_norm_file = path.join('../data','processed_data','log_normalizedCounts_norm.csv')
final_metadata_file = path.join('../data','processed_data','metadata.tsv')
final_metadata_all_file = path.join('../data','interim','metadata_qc_part2_all.tsv')


DF_log_tpm_final.to_csv(logTPM_qc_file)
DF_log_tpm_norm.to_csv(logTPM_norm_file)
DF_log_normalizedCounts.to_csv(logNormalizedCounts_qc_file)
DF_log_normalizedCounts_norm.to_csv(logNormalizedCounts_norm_file)
DF_metadata_final.to_csv(final_metadata_file, sep='\t')
DF_metadata_all.to_csv(final_metadata_all_file, sep='\t')

In [ ]:
final_metadata_file = path.join('../data','processed_data','metadata.tsv')
DF_metadata_final.to_csv(final_metadata_file, sep='\t')

In [ ]:
DF_metadata_final['condition'].unique().size